# Automating a Ghana Agrivoltaics Data Pipeline

This notebook is the code-along companion for the 3-hour workshop. We take a
year of raw agrivoltaics **sensor logs** and turn them, one command at a time,
into a trusted answer to a real question:

> **Do the raised solar panels keep the crops cooler - while still generating power?**

We build the pipeline in stages: **load -> reshape -> contract -> clean ->
validate -> insights.** We never trust the data until it has earned it.

## Dataset

- Kaggle: https://www.kaggle.com/datasets/responsibleailab/agrivoltaic-dataset-ghana
- Provider: Responsible AI Lab, KNUST  -  License: CC-BY 4.0

The files in `data/raw/` are **environmental sensor telemetry**, not crop yields:

- 5 Excel workbooks, **one per month** (May-Oct 2024)
- **~30 sheets per workbook - one sheet per day** (e.g. `1 08 24`)
- each day-sheet is a wide grid: **plot code** over **measurement type**, with
  **time** down the side, logged every 5 minutes
- three plots: `AO` = open control field, `AG` = agrivoltaic (raised panels),
  `PO` = ground-mounted PV; `WS` = weather station

## Module 1: Load the Raw Sheets

**Big idea:** the first job of a pipeline is to reliably *find and open* its
inputs. Here every workbook is a month and every sheet is a day - so we list the
files **and** the sheets inside them.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("..").resolve()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
REPORTS_DIR = ROOT / "outputs" / "reports"
CHARTS_DIR = ROOT / "outputs" / "charts"

for directory in [PROCESSED_DIR, REPORTS_DIR, CHARTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("raw data dir:", RAW_DIR)

In [ ]:
SUPPORTED = {".csv", ".xls", ".xlsx"}

workbooks = sorted(p for p in RAW_DIR.glob("*") if p.suffix.lower() in SUPPORTED)
print(f"{len(workbooks)} workbook(s):")
for path in workbooks:
    print("  ", path.name)

# Peek inside the first workbook: one sheet per day
sheets = pd.ExcelFile(workbooks[0]).sheet_names
print(f"\n{workbooks[0].name} has {len(sheets)} sheets, e.g. {sheets[:5]}")

In [ ]:
# Read ONE day-sheet exactly as recorded: header=None keeps every messy row.
raw = pd.read_excel(workbooks[0], sheet_name=sheets[0], header=None)
print("shape:", raw.shape)
raw.iloc[:5, :8]

### Checkpoint

- All five workbooks are listed.
- You can read one day-sheet and see the two header rows (plot code, then
  measurement) with the time column on the left.
- Nothing has been cleaned yet - that is deliberate.

## Module 2: Reshape & Profile

**Big idea:** that wide two-header grid is hard to work with. We reshape every
sheet into **tidy long rows** (one reading per row), then **profile** what we
have. The header row sometimes drifts, so we *detect* it instead of assuming it.

In [ ]:
import re

MEASUREMENT_LABELS = {"Irr (W/m2)", "T (oC)", "RH (%)", "P (mm)"}

def detect_header_rows(raw):
    """Find (plot-code row, measurement row) by matching the measurement labels."""
    for i in range(min(6, len(raw))):
        labels = {str(v).strip() for v in raw.iloc[i] if isinstance(v, str)}
        if labels & MEASUREMENT_LABELS:
            return i - 1, i
    raise ValueError("No measurement header row found")

def parse_sheet_date(sheet_name):
    """'1 08 24' or '1_07_24' -> a real date."""
    tokens = [t for t in re.split(r"[ _]+", sheet_name.strip()) if t]
    if len(tokens) < 3 or not all(t.isdigit() for t in tokens[:3]):
        return None
    day, month, year = (int(t) for t in tokens[:3])
    year += 2000 if year < 100 else 0
    return pd.Timestamp(year=year, month=month, day=day)

print("header rows:", detect_header_rows(raw))
print("sheet date :", parse_sheet_date(sheets[0]).date())

In [ ]:
def tidy_sheet(raw, sheet_date):
    """Reshape one wide day-sheet into long rows."""
    plot_row, meas_row = detect_header_rows(raw)
    plot_codes = raw.iloc[plot_row].ffill()        # carry codes across merges
    measurements = raw.iloc[meas_row]
    body = raw.iloc[meas_row + 1:].reset_index(drop=True)
    raw_time = body.iloc[:, 0]

    columns = []
    for col in range(1, raw.shape[1]):
        plot, meas = plot_codes.iloc[col], measurements.iloc[col]
        if not (isinstance(plot, str) and isinstance(meas, str)):
            continue
        if meas.strip() not in MEASUREMENT_LABELS:
            continue
        columns.append(pd.DataFrame({
            "date": sheet_date,
            "raw_time": raw_time.values,
            "plot_code": plot.strip(),
            "measurement": meas.strip(),
            "value": body.iloc[:, col].values,
        }))
    return pd.concat(columns, ignore_index=True)

tidy_sheet(raw, parse_sheet_date(sheets[0])).head()

In [ ]:
# Build the full tidy table across every day in every workbook.
# (This is the slow cell - it reads ~150 sheets. Give it ~30 seconds.)
frames = []
for workbook in workbooks:
    for sheet in pd.ExcelFile(workbook).sheet_names:
        sheet_date = parse_sheet_date(sheet)
        if sheet_date is None:
            continue
        raw_sheet = pd.read_excel(workbook, sheet_name=sheet, header=None)
        frames.append(tidy_sheet(raw_sheet, sheet_date))

long_table = pd.concat(frames, ignore_index=True)
print(f"{len(long_table):,} readings")
long_table.head()

In [ ]:
def profile_table(df):
    return pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "missing_pct": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
        "unique_values": [df[c].nunique(dropna=True) for c in df.columns],
    })

profile = profile_table(long_table)
profile.to_csv(REPORTS_DIR / "profile_readings.csv", index=False)
profile

### Your turn / Checkpoint

- Every day-sheet reshapes into tidy rows; the full table holds ~1.1M readings.
- Read the profile: **how much of `value` is missing?** (Sensor gaps, not errors.)
- A `profile_readings.csv` is saved in `outputs/reports/`.

## Module 3: Define the Data Contract

**Big idea:** the cryptic plot codes carry the whole experiment. We encode that
meaning once - prefix to treatment, label to unit - so the rest of the pipeline
speaks `agrivoltaic`, not `AG`.

In [ ]:
MEASUREMENT_NAMES = {"Irr (W/m2)": "irradiance", "T (oC)": "temperature",
                     "RH (%)": "humidity", "P (mm)": "rainfall"}
MEASUREMENT_UNITS = {"irradiance": "W/m2", "temperature": "C",
                     "humidity": "%", "rainfall": "mm"}
TREATMENT_BY_PREFIX = {"AG": "agrivoltaic", "AO": "open_sun_control",
                       "PO": "ground_mounted_pv", "WS": "ambient"}
ALLOWED_PREFIXES = set(TREATMENT_BY_PREFIX)
ALLOWED_RANGES = {"irradiance": (0, 1500), "temperature": (5, 60),
                  "humidity": (0, 100), "rainfall": (0, 300)}

def parse_plot_code(code):
    base, _, replicate = code.strip().partition(" ")
    prefix, _, station = base.partition("-")
    return {"prefix": prefix, "station": station or None,
            "replicate": replicate or None,
            "treatment": TREATMENT_BY_PREFIX.get(prefix, "unknown")}

parse_plot_code("AG-PV P3")

In [ ]:
def apply_contract(long_table):
    out = long_table.copy()
    parts = out["plot_code"].map(parse_plot_code)
    out["treatment"] = [p["treatment"] for p in parts]
    out["station"] = [p["station"] for p in parts]
    out["replicate"] = [p["replicate"] for p in parts]
    out["measurement"] = out["measurement"].map(MEASUREMENT_NAMES).fillna(
        out["measurement"])
    out["unit"] = out["measurement"].map(MEASUREMENT_UNITS)
    return out

contracted = apply_contract(long_table)
contracted[["plot_code", "treatment", "station", "measurement", "unit"]] \
    .drop_duplicates().head(10)

### Checkpoint

- Every plot prefix maps to a known treatment (`AG/AO/PO/WS`).
- Each reading now carries a `treatment`, `station`, `measurement`, and `unit`.

## Module 4: Clean and Standardize

**Big idea:** make times real and values numeric - without erasing evidence.
Junk like `11:34 - Need network restart` becomes `NaT`; un-parseable numbers
become `NaN`. We expose failure instead of hiding it.

In [ ]:
def clean_long_table(long_table):
    cleaned = long_table.copy()
    cleaned["date"] = pd.to_datetime(cleaned["date"], errors="coerce")
    stamp = (cleaned["date"].dt.strftime("%Y-%m-%d")
             + " " + cleaned["raw_time"].astype(str))
    cleaned["timestamp"] = pd.to_datetime(stamp, errors="coerce")
    cleaned["value"] = pd.to_numeric(cleaned["value"], errors="coerce")
    return cleaned

cleaned = clean_long_table(contracted)
cleaned.head()

In [ ]:
print("unparseable timestamps:", int(cleaned["timestamp"].isna().sum()))
print("non-numeric / missing values:", int(cleaned["value"].isna().sum()))
cleaned.dtypes

### Checkpoint

- `timestamp` is a real datetime (or a visible `NaT`).
- `value` is numeric (or a visible `NaN`).
- The original `long_table` is untouched - we worked on a copy.

## Module 5: Validate Before Trusting

**Big idea:** validation decides what may be published. We separate **structural
failures** (an unknown plot code -> STOP) from **data-quality issues** (a few
out-of-range spikes -> quarantine and warn).

In [ ]:
def validate_known_plots(cleaned):
    prefixes = cleaned["plot_code"].str.split("-").str[0].str.split(" ").str[0]
    unknown = sorted(set(prefixes.dropna()) - ALLOWED_PREFIXES)
    return [f"Unknown plot prefixes: {unknown}"] if unknown else []

def validate_ranges(cleaned):
    errors = []
    for name, (low, high) in ALLOWED_RANGES.items():
        values = cleaned.loc[cleaned["measurement"] == name, "value"]
        bad = values[(values < low) | (values > high)]
        if not bad.empty:
            errors.append(f"{name}: {len(bad)} values outside [{low}, {high}]")
    return errors

def validate_timestamps(cleaned):
    missing = int(cleaned["timestamp"].isna().sum())
    return [f"{missing} rows have an unparseable timestamp"] if missing else []

hard_errors = validate_known_plots(cleaned)
warnings = validate_ranges(cleaned) + validate_timestamps(cleaned)

report = pd.DataFrame(
    [{"status": "error", "message": m} for m in hard_errors]
    + [{"status": "warning", "message": m} for m in warnings]
    or [{"status": "ok", "message": "Validation passed"}]
)
report.to_csv(REPORTS_DIR / "validation_report.csv", index=False)
report

In [ ]:
# Data-quality issues don't stop the run - we quarantine the impossible readings
# (set them to NaN) but keep their rows as evidence.
def quarantine_out_of_range(cleaned):
    out = cleaned.copy()
    for name, (low, high) in ALLOWED_RANGES.items():
        mask = out["measurement"] == name
        bad = mask & ((out["value"] < low) | (out["value"] > high))
        out.loc[bad, "value"] = float("nan")
    return out

trusted = quarantine_out_of_range(cleaned)
print("rows kept:", f"{len(trusted):,}")

### Your turn / Checkpoint

- A `validation_report.csv` is saved with every issue named.
- **Break it on purpose:** rename a plot to `ZZ-XX` and re-run
  `validate_known_plots` - confirm it raises a structural error.
- A 350 reading is quarantined (becomes `NaN`), not silently published.

## Module 6: Transform Into Insights

**Big idea:** 1.1M readings can't be read directly. We summarise to a meaningful
**grain**, then compare the agrivoltaic plot against the right open reference at
**midday** - when sun, heat, and the panels' effect all peak.

In [ ]:
daily = (
    trusted.dropna(subset=["timestamp", "value"])
    .groupby(["date", "treatment", "station", "measurement"], dropna=False)
    .agg(observations=("value", "count"),
         mean_value=("value", "mean"),
         min_value=("value", "min"),
         max_value=("value", "max"))
    .reset_index()
)
daily.to_csv(PROCESSED_DIR / "daily_plot_summary.csv", index=False)
daily.head()

In [ ]:
# Each measurement is compared with the baseline that actually exists:
# temp/humidity vs the open control field, irradiance vs the full-sun PV plot.
BASELINE_BY_MEASUREMENT = {"temperature": "open_sun_control",
                           "humidity": "open_sun_control",
                           "irradiance": "ground_mounted_pv"}

midday = trusted.dropna(subset=["timestamp", "value"]).set_index("timestamp")
midday = midday.between_time("11:00", "14:00").reset_index()
means = midday.groupby(["measurement", "treatment"])["value"].mean().to_dict()

rows = []
for measurement, baseline in BASELINE_BY_MEASUREMENT.items():
    agrivoltaic = means.get((measurement, "agrivoltaic"))
    reference = means.get((measurement, baseline))
    if agrivoltaic is None or reference is None:
        continue
    diff = agrivoltaic - reference
    rows.append({"measurement": measurement,
                 "agrivoltaic": round(agrivoltaic, 2),
                 "reference": baseline,
                 "reference_value": round(reference, 2),
                 "difference": round(diff, 2),
                 "difference_pct": round(diff / reference * 100, 2)})

comparison = pd.DataFrame(rows)
comparison.to_csv(PROCESSED_DIR / "midday_microclimate.csv", index=False)
comparison

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(comparison), figsize=(11, 4))
for ax, (_, row) in zip(axes, comparison.iterrows()):
    ax.bar(["Open reference", "Agrivoltaic"],
           [row["reference_value"], row["agrivoltaic"]],
           color=["#F7C948", "#54D17A"])
    ax.set_title(f"{row['measurement'].title()} ({row['difference_pct']:+}%)")
    ax.grid(axis="y", alpha=0.3)
fig.suptitle("Midday microclimate: agrivoltaic vs open reference")
fig.tight_layout()
fig.savefig(CHARTS_DIR / "midday_microclimate.png", dpi=150)
plt.show()

### Checkpoint

You should now have, in this project:

```
data/processed/daily_plot_summary.csv
data/processed/midday_microclimate.csv
outputs/reports/profile_readings.csv
outputs/reports/validation_report.csv
outputs/charts/midday_microclimate.png
```

And the headline: midday is markedly **cooler under the panels** than in the
open field, while the panels still receive strong irradiance.

## Wrap-Up: Responsible Interpretation

Write 5-7 sentences answering:

1. What does the comparison actually show?
2. Why is a cooler, drier microclimate interesting for crops - and what does it
   *not* prove (e.g. yield)?
3. Why is irradiance compared against the ground-mounted PV, not the control field?
4. What are this pilot dataset's limits (one site, sensor gaps, midday window)?
5. What extra data would strengthen the conclusion?

Carry forward with every number: **source, licence, grain, baseline, validation
status, and limitations.** A reproducible result can still mislead if you drop
its context.

> The whole pipeline can be rebuilt with one command: `python src/run_pipeline.py`